# 1.4 — Feature Engineering
**Continued from:** `1.1-jp-eda.ipynb`  
**Purpose:** Explore feature transformations (binning, encoding, scaling, oversampling), select a final feature set, and build a Vertex AI training pipeline using KFP.

---
**Notebook Sections**
1. [Setup](#1-setup)
2. [Experiment 1 — Age Binning Comparison](#2-experiment-1)
3. [Experiment 2 — Scaling](#3-experiment-2)
4. [Experiment 3 — Final Feature Set + Multi-Model Evaluation](#4-experiment-3)
5. [GCP / Vertex AI Integration](#5-gcp)
6. [KFP Pipeline Components](#6-components)
7. [Pipeline Assembly & Deployment](#7-pipeline)


---
## 1. Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from google.cloud import aiplatform
from google.cloud import storage
from sklearn.preprocessing import OneHotEncoder

In [ ]:
df = pd.read_csv('../data/interim/1.2-edited-data.csv', keep_default_na=False)

df_4 = df.copy().drop(columns=["weight_kg"])

In [ ]:
# ---------------------------------------------------------------------------
# Preprocessing helpers
# Used by the exploratory experiment cells below.
# NOTE: KFP components are self-contained closures and duplicate this logic
#       intentionally — they cannot reference notebook-scope functions.
# ---------------------------------------------------------------------------

def clean_prescribed_features(df: pd.DataFrame) -> pd.DataFrame:
    """Coerce medications_prescribed to float and derive is_prescribed flag.
    Also coerces number_of_prior_visits to float.
    Operates on a copy and returns it."""
    df = df.copy()
    df["medications_prescribed"] = df["medications_prescribed"].replace("", pd.NA).astype(float)
    df["is_prescribed"] = df["medications_prescribed"].apply(lambda x: 1 if x > 0 else 0)
    df["number_of_prior_visits"] = df["number_of_prior_visits"].replace("", pd.NA).astype(float)
    return df


def add_length_of_stay_score(df: pd.DataFrame) -> pd.DataFrame:
    """Converts length_of_stay (days) to an ordinal risk score (1–7)."""
    df = df.copy()
    def _score(x):
        if x <= 1: return 1
        if x <= 2: return 2
        if x <= 3: return 3
        if x <= 6: return 4
        if x <= 14: return 5
        return 7
    df["length_of_stay_score"] = df["length_of_stay"].apply(_score)
    return df


### Goals
- Evaluate binning strategies for `age` and `length_of_stay`
- Determine how to handle sparse/missing values in `medications_prescribed` and `number_of_prior_visits`
- Compare model performance across feature engineering strategies
- Finalise a feature set for the production training pipeline

---
## 2. Experiment 1 — Age Binning Comparison

Compare raw `age` vs. age bins as a feature using a Random Forest baseline (no scaling, no oversampling).

In [ ]:
## binning age

age_bins_1 = [0, 18, 25, 40, 65, 80, 100]

df_4["age_bin_1"] = pd.cut(df_4["age"], bins=age_bins_1)

df_4["age_bin_1"].value_counts()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

model = RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        random_state=42,
    )

In [ ]:
df_bins = df_4.copy().drop(columns=["age"])
df_nonbins = df_4.copy().drop(columns=["age_bin_1"])

df_bins = pd.get_dummies(df_bins, drop_first=True)
df_nonbins = pd.get_dummies(df_nonbins, drop_first=True)

In [ ]:
X_bins_train, X_bins_test, y_bins_train, y_bins_test = train_test_split(
    df_bins.drop(columns=["target"]),
    df_bins["target"],
    test_size=0.2,
    random_state=42,
)

model.fit(X_bins_train, y_bins_train)

preds = model.predict(X_bins_test)

print(classification_report(y_bins_test, preds))

print(confusion_matrix(y_bins_test, preds))

In [ ]:
X_nonbins_train, X_nonbins_test, y_nonbins_train, y_nonbins_test = train_test_split(
    df_nonbins.drop(columns=["target"]),
    df_nonbins["target"],
    test_size=0.2,
    random_state=42,
)
model.fit(X_nonbins_train, y_nonbins_train)

preds_nonbins = model.predict(X_nonbins_test)

print(classification_report(y_nonbins_test, preds_nonbins))

print(confusion_matrix(y_nonbins_test, preds_nonbins))

### Observations — Experiment 1

- Age binning performs slightly better than raw age on an unscaled, unbalanced dataset.
- Both configurations still reflect the class imbalance problem — the next step is to normalise features and then re-evaluate.

---
## 3. Experiment 2 — MinMax Scaling

Apply `MinMaxScaler` to the binned and non-binned splits to see if normalisation improves performance.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

cols_to_scale_binned = ["height_m", "bmi", "adjusted_weight_kg", "length_of_stay"]
cols_to_scale_non_binned = ["height_m", "bmi", "adjusted_weight_kg", "length_of_stay", "age"]

bins_scaler = MinMaxScaler()
non_bins_scaler = MinMaxScaler()

X_bins_train[cols_to_scale_binned] = bins_scaler.fit_transform(X_bins_train[cols_to_scale_binned])
X_nonbins_train[cols_to_scale_non_binned] = non_bins_scaler.fit_transform(X_nonbins_train[cols_to_scale_non_binned])

In [ ]:
model.fit(X_bins_train, y_bins_train)
preds_scaled_bins = model.predict(X_bins_test)
print(classification_report(y_bins_test, preds_scaled_bins))


model.fit(X_nonbins_train, y_nonbins_train)
preds_scaled_nonbins = model.predict(X_nonbins_test)
print(classification_report(y_nonbins_test, preds_scaled_nonbins))

## Experiment 3 — Revised Feature Set

Rebuilding from `df_4` with a broader set of transformations:
- Labelled age bins instead of numeric bins
- Ordinal `length_of_stay_score` instead of categorical bins
- SMOTE oversampling
- Multiple classifiers for comparison

In [ ]:
df = df_4.copy()

df = df.drop(columns = ["age"])

In [ ]:
length_of_stay_categories = {
    "0-3": (0, 3),
    "4-7": (4, 7),
    "8-14": (8, 14),
    "15-30": (15, 30),
    "31+": (31, np.inf)
}

df["length_of_stay_cat"] = pd.cut(df["length_of_stay"], bins=[0, 3, 7, 14, 30, np.inf], labels=length_of_stay_categories.keys(), right=False)   

In [ ]:
df = clean_prescribed_features(df)


In [ ]:
df_features = df[[
    "gender",
    "age_bin_1",
    "height_m",
    "bmi",
    "adjusted_weight_kg",
    "length_of_stay_cat",
    "is_prescribed",
    "number_of_prior_visits",
    "smoker",
    "has_diabetes",
    "has_hypertension",
    "exercise_frequency",
    "diet_type",
    "type_of_treatment",
    "target"
]]

df_features = df_features.dropna(inplace=False)

In [ ]:
df_feats = df_features.copy().drop(columns=["target"])
target = df_features["target"]
df_features = pd.get_dummies(df_features, drop_first=True)

df_features

In [ ]:
from sklearn.preprocessing import StandardScaler

cols_to_scale = ["height_m", "bmi", "adjusted_weight_kg", "number_of_prior_visits"]

df_feats = pd.get_dummies(df_feats, drop_first=True)

df_feats_smote = df_feats.copy()


X_train, X_test, y_train, y_test = train_test_split(
    df_feats,
    target,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])

model.fit(X_train, y_train)

preds = model.predict(X_test)
print(classification_report(y_test, preds))

In [ ]:
from imblearn.over_sampling import SMOTE, RandomOverSampler

ros = RandomOverSampler(random_state=42)

X_train_ros, X_test_ros, y_train_ros, y_test_ros = train_test_split(df_feats_smote, target, test_size=0.2, random_state=42, stratify=target)

X_train_ros, y_train_ros = ros.fit_resample(X_train_ros, y_train_ros)

X_train_ros[cols_to_scale] = scaler.fit_transform(X_train_ros[cols_to_scale])
model.fit(X_train_ros, y_train_ros)

preds_ros = model.predict(X_test_ros)
print(classification_report(y_test_ros, preds_ros))

print(confusion_matrix(y_test_ros, preds_ros))


In [ ]:
from sklearn.ensemble import AdaBoostClassifier

ada_model = AdaBoostClassifier(
    n_estimators=250,
    learning_rate=0.01,
    random_state=42,
)   

ada_model.fit(X_train_ros, y_train_ros)
preds_ada = ada_model.predict(X_test_ros)
print(classification_report(y_test_ros, preds_ada))
    

### Observations — Experiment 2

Random Forest is not predicting the minority class even with balanced data. The scaler was fit on the original training set and not re-applied to the ROS-resampled features — investigating in Experiment 3.

In [ ]:
X_train_ros

### Models to Evaluate

1. Logistic Regression — baseline
2. Random Forest — tune hyperparameters
3. AdaBoost
4. Gradient Boosting (XGBoost-style via sklearn)

### Experiment 3 — Revised Encodings

Switching `length_of_stay` to an ordinal score and using labelled age bins to improve interpretability.

In [ ]:
df = df_4.copy()

age_bins_label = {
    "0-18": (0, 18),
    "19-25": (19, 25),
    "26-40": (26, 40),
    "41-65": (41, 65),
    "66-80": (66, 80),
    "81+": (81, np.inf)
}

df["age_bin_label"] = pd.cut(df["age"], bins=[0, 18, 25, 40, 65, 80, np.inf], labels=age_bins_label.keys(), right=False)

In [ ]:
df.drop(columns=["age_bin_1"], inplace=True)

In [ ]:
df = clean_prescribed_features(df)


In [ ]:
df = add_length_of_stay_score(df)


In [ ]:
df = df.dropna(inplace=False)
target = df["target"]
df_features = df.drop(columns=["age", "length_of_stay", "medications_prescribed", "patient_id", "target"])


In [ ]:
df_features = pd.get_dummies(df_features, drop_first=True)

cols_to_scale = ["height_m", "bmi", "adjusted_weight_kg", "number_of_prior_visits", "length_of_stay_score"]


X_train, X_test, y_train, y_test = train_test_split(
    df_features,
    target,
    test_size=0.2,
    random_state=42,
    stratify=target
)


smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

scaler = StandardScaler()

X_train_smote[cols_to_scale] = scaler.fit_transform(X_train_smote[cols_to_scale])
X_train[cols_to_scale] = scaler.transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

rf_model = RandomForestClassifier(
    n_estimators=250,
    max_depth=8,
    random_state=42,
)

ada_model = AdaBoostClassifier(
    n_estimators=250,
    learning_rate=0.001,
    random_state=42,
)
log_reg_model = LogisticRegression(
    random_state=42,
    max_iter=1000,
)

xgb_model = GradientBoostingClassifier(
    n_estimators=250,
    learning_rate=0.01,
    max_depth=5,
    random_state=42,
)

rf_model.fit(X_train_smote, y_train_smote)
ada_model.fit(X_train_smote, y_train_smote)
log_reg_model.fit(X_train_smote, y_train_smote)
xgb_model.fit(X_train_smote, y_train_smote)

In [ ]:
rf_preds = rf_model.predict(X_test)
ada_preds = ada_model.predict(X_test)

print("Random Forest Classification Report:")
print(classification_report(y_test, rf_preds))

print("AdaBoost Classification Report:")
print(classification_report(y_test, ada_preds))

log_reg_preds = log_reg_model.predict(X_test)
print("Logistic Regression Classification Report:")
print(classification_report(y_test, log_reg_preds))

xgb_preds = xgb_model.predict(X_test)
print("XGBoost Classification Report:")
print(classification_report(y_test, xgb_preds))

### Results — Experiment 3

> **Note:** Test set must be transformed with the scaler fit on training data — not re-fit. This is handled correctly in the KFP `apply_preprocessing` component below.

---
## GCP / Vertex AI Integration

Feature engineering is baked into the training pipeline rather than versioned as a dataset transformation.  
The pipeline handles: data validation → train/val split → oversampling → preprocessing (fit + apply) → model training → evaluation.

In [ ]:
""" first going to make a v1.4 dataset that just removes weight because thats the only constant and will be
going forward i believe"""

import sys
sys.path.insert(0, "..")
from scripts.gcs_utils import log_dataset_to_gcs, log_pipeline_run
from pathlib import Path

df = pd.read_csv('../data/interim/1.2-edited-data.csv', keep_default_na=False)

df_4 = df.copy().drop(columns=["weight_kg"])

df_4.to_csv('../data/interim/1.4-edited-data.csv', index=False)


In [ ]:
PROJECT_ID = "readmission-543-project"
LOCATION = "us-central1"
BUCKET_ROOT_URI = "gs://readmission-bucket"


aiplatform.init(project=PROJECT_ID, location=LOCATION)

log_dataset_to_gcs(
    DATASET_LOCAL_PATH=Path("../data/interim/1.4-edited-data.csv"),
    VERSION_ID="v1.4",
    BUCKET_ROOT_URI=BUCKET_ROOT_URI,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    log_experiment=True,
    EXPERIMENT_NAME="readmissions-data-versions",
    description="Dataset with initial cleaning+formatting, as well as weight_kg feature removed.",
    tags=["initial-clean", "v1.4", "weight-removed"],
    resume_run=True,
)

In [ ]:
df_4

---
## 6. KFP Pipeline Components

Each component below is a self-contained KFP v2 `@component` decorated function. They are combined into the pipeline in Section 7.

### Component 1 — `load_validate_data`

Reads a CSV from GCS, validates required columns and target distribution, and passes the dataset downstream.

In [ ]:
from kfp.v2 import dsl
from kfp.v2.dsl import component, Output, Dataset, Input, Model, Metrics, Artifact
from google.cloud import aiplatform


@component(packages_to_install=["pandas", "numpy", "fsspec", "gcsfs"])
def load_validate_data(input_dataset_path: str, output_dataset: Output[Dataset]):
    import pandas as pd
    import numpy as np

    df = pd.read_csv(input_dataset_path)

    # --- Basic dataset validation ---
    if df.empty:
        raise ValueError("Input dataset is empty")

    print(f"Dataset shape: {df.shape}")

    # Drop accidental CSV index columns if present
    df = df.loc[:, ~df.columns.str.contains(r"^Unnamed")]

    # --- Required pipeline columns ---
    target_col = "target"
    id_col = "patient_id"

    required_cols = [target_col, id_col]
    missing_required = [col for col in required_cols if col not in df.columns]
    if missing_required:
        raise ValueError(f"Missing required columns: {missing_required}")

    # --- Target validation ---
    print("Target distribution:")
    print(df[target_col].value_counts(dropna=False))

    if df[target_col].dropna().nunique() < 2:
        raise ValueError(
            f"Target column '{target_col}' must contain at least two classes"
        )

    # --- Save output ---
    df.to_csv(output_dataset.path, index=False)

### Component 2 — `split_data`

Stratified train/validation split. Preserves `patient_id` throughout.

In [ ]:
@component(packages_to_install=["pandas", "scikit-learn"])
def split_data(
    input_dataset: Input[Dataset],
    train_dataset: Output[Dataset],
    validation_dataset: Output[Dataset],
    test_size: float = 0.2,
    random_state: int = 42,
):

    from sklearn.model_selection import train_test_split
    import pandas as pd

    df = pd.read_csv(input_dataset.path)

    target_col = "target"
    id_col = "patient_id"

    X = df.drop(columns=[target_col, id_col])
    y = df[target_col]
    ids = df[id_col]

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )

    train_df = pd.concat([X_train, y_train, ids.loc[X_train.index]], axis=1)
    val_df = pd.concat([X_val, y_val, ids.loc[X_val.index]], axis=1)

    train_df.to_csv(train_dataset.path, index=False)
    val_df.to_csv(validation_dataset.path, index=False)

    print(f"Training set shape: {train_df.shape}")
    print(f"Validation set shape: {val_df.shape}")

    print("Training target distribution:")
    print(train_df[target_col].value_counts(normalize=True))

    print("Validation target distribution:")
    print(val_df[target_col].value_counts(normalize=True))

### Component 3 — `oversample_training`

Applies `RandomOverSampler` to the training split only to address class imbalance.

In [ ]:
@component(
    packages_to_install=[
        "pandas",
        "numpy",
        "scikit-learn",
        "imbalanced-learn"
    ]
)
def oversample_training(
    input_dataset: Input[Dataset],
    output_dataset: Output[Dataset],
    target_col: str = "target",
    id_col: str = "patient_id",
    random_state: int = 42
):
    import pandas as pd
    from imblearn.over_sampling import RandomOverSampler

    # --- Load dataset ---
    df = pd.read_csv(input_dataset.path)

    print(f"Input training dataset shape: {df.shape}")

    # --- Validate required columns ---
    if target_col not in df.columns:
        raise ValueError(f"Missing target column: {target_col}")

    if id_col not in df.columns:
        raise ValueError(f"Missing ID column: {id_col}")

    # --- Check class distribution before ---
    print("Class distribution BEFORE oversampling:")
    print(df[target_col].value_counts(dropna=False))

    # --- Separate components ---
    # Note: ID is kept as part of the dataset but not used for modeling logic
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    # --- Apply Random Oversampling ---
    ros = RandomOverSampler(random_state=random_state)

    X_resampled, y_resampled = ros.fit_resample(X, y)

    # --- Reconstruct dataset ---
    resampled_df = X_resampled.copy()
    resampled_df[target_col] = y_resampled

    # --- Log results ---
    print("Class distribution AFTER oversampling:")
    print(resampled_df[target_col].value_counts(dropna=False))

    print(f"Original dataset shape: {df.shape}")
    print(f"Oversampled dataset shape: {resampled_df.shape}")

    # --- Save output ---
    resampled_df.to_csv(output_dataset.path, index=False)

### Component 4 — `fit_apply_preprocessing`

Fits all preprocessing steps (imputation, scaling, one-hot encoding) on the training split and saves artifacts for reuse by the validation component.

In [ ]:
@component(packages_to_install=["pandas", "numpy", "scikit-learn", "joblib"])
def fit_apply_preprocessing(
    input_dataset: Input[Dataset],
    output_dataset: Output[Dataset],
    preprocessing_artifacts: Output[Artifact],
    target_col: str = "target",
    id_col: str = "patient_id",
):
    import os
    import json
    import joblib
    import numpy as np
    import pandas as pd
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler

    # -----------------------------
    # Load training dataset
    # -----------------------------
    df = pd.read_csv(input_dataset.path)

    print(f"Input training dataset shape: {df.shape}")

    # -----------------------------
    # Separate ID and target
    # Keep them outside the fitted preprocessing logic
    # -----------------------------
    ids = df[[id_col]].copy()
    y = df[[target_col]].copy()
    X = df.drop(columns=[id_col, target_col]).copy()

    # -----------------------------
    # Fixed-rule preprocessing
    # -----------------------------

    X["age_group"] = pd.cut(
        X["age"],
        bins=[0, 18, 25, 40, 65, 80, np.inf],
        labels=["0-18", "19-25", "26-40", "41-65", "66-80", "81+"],
        right=False,
    )
    X = X.drop(columns=["age"])

    X["medications_prescribed"] = X["medications_prescribed"].replace("", pd.NA)
    X["medications_prescribed"] = X["medications_prescribed"].astype(float)

    X["medications_prescribed"] = X["medications_prescribed"].apply(
        lambda x: 1 if x > 0 else 0
    )

    X["number_of_prior_visits"] = X["number_of_prior_visits"].replace("", pd.NA)
    X["number_of_prior_visits"] = X["number_of_prior_visits"].astype(float)

    X["length_of_stay_score"] = X["length_of_stay"].apply(
        lambda x: (
            1
            if x <= 1
            else (
                2
                if x <= 2
                else (3 if x <= 3 else (4 if x <= 6 else (5 if x <= 14 else 7)))
            )
        )
    )
    X.drop(columns=["length_of_stay"], inplace=True)
    
    num_cols = ["height_m", "bmi", "adjusted_weight_kg", "number_of_prior_visits", "length_of_stay_score"]
    cat_cols = X.select_dtypes(include=["object", "category", "str"]).columns.tolist()
    # boolean handling taken out becasue fml add it back later ^_^

    # -----------------------------
    # Learned preprocessing
    # -----------------------------
    
    #impute median on numerics
    
    median_imputer = SimpleImputer(strategy="median", missing_values= pd.NA)
    mode_imputer = SimpleImputer(strategy="most_frequent", missing_values= pd.NA)
    
    
    X[num_cols] = median_imputer.fit_transform(X[num_cols])
    X[cat_cols] = mode_imputer.fit_transform(X[cat_cols])
    
    #standardize numeric cols
    scaler = StandardScaler()
    X[num_cols] = scaler.fit_transform(X[num_cols])
    
    # one hot encode categories
    X = pd.get_dummies(X, drop_first=True)
    
    # Reconstruct final training dataset
    # -----------------------------
    prepared_df = pd.concat([ids, X, y], axis=1)

    print(f"Prepared training dataset shape: {prepared_df.shape}")

    # -----------------------------
    # Save transformed dataset
    # -----------------------------
    prepared_df.to_csv(output_dataset.path, index=False)

    # -----------------------------
    # Save preprocessing artifacts
    # -----------------------------
    os.makedirs(preprocessing_artifacts.path, exist_ok=True)

    # Save encoder
    joblib.dump(
        scaler, os.path.join(preprocessing_artifacts.path, "standard_scaler.joblib")
    )
    
    # Save imputers
    
    joblib.dump(
        median_imputer, os.path.join(preprocessing_artifacts.path, "median_imputer.joblib")
    )
    joblib.dump(
        mode_imputer, os.path.join(preprocessing_artifacts.path, "mode_imputer.joblib")
    )
    
    # Save preprocessing metadata
    metadata = {
        "target_col": 'target',
        "id_col": 'patient_id',
        "num_cols": num_cols,
        "categorical_cols": cat_cols,
        "output_feature_columns": X.columns.tolist(),

    }
    with open(
        os.path.join(preprocessing_artifacts.path, "preprocessing_metadata.json"), "w"
    ) as f:
        json.dump(metadata, f, indent=2)

    print("Preprocessing artifacts saved:")
    print(os.listdir(preprocessing_artifacts.path))

### Component 5 — `apply_preprocessing`

Applies the saved preprocessing artifacts from Component 4 to the validation split. Realigns feature columns to match the training schema.

In [ ]:
@component(
    packages_to_install=[
        "pandas",
        "numpy",
        "scikit-learn",
        "joblib"
    ]
)
def apply_preprocessing(
    input_dataset: Input[Dataset],
    preprocessing_artifacts: Input[Artifact],
    output_dataset: Output[Dataset],
    target_col: str = "target",
    id_col: str = "patient_id"
):
    import os
    import json
    import joblib
    import numpy as np
    import pandas as pd

    # -----------------------------
    # Load validation dataset
    # -----------------------------
    df = pd.read_csv(input_dataset.path)

    print(f"Input validation dataset shape: {df.shape}")

    # -----------------------------
    # Load preprocessing artifacts
    # -----------------------------
    metadata_path = os.path.join(preprocessing_artifacts.path, "preprocessing_metadata.json")
    median_imputer_path = os.path.join(preprocessing_artifacts.path, "median_imputer.joblib")
    mode_imputer_path = os.path.join(preprocessing_artifacts.path, "mode_imputer.joblib")
    scaler_path = os.path.join(preprocessing_artifacts.path, "standard_scaler.joblib")

    if not os.path.exists(metadata_path):
        raise ValueError(f"Missing preprocessing metadata file: {metadata_path}")

    if not os.path.exists(scaler_path):
        raise ValueError(f"Missing scaler artifact file: {scaler_path}")

    with open(metadata_path, "r") as f:
        metadata = json.load(f)

    median_imputer = joblib.load(median_imputer_path)
    mode_imputer = joblib.load(mode_imputer_path)
    scaler = joblib.load(scaler_path)
    
    
    num_cols = metadata["num_cols"]
    categorical_cols = metadata["categorical_cols"]
    expected_output_cols = metadata["output_feature_columns"]

    print("Numeric columns to impute and scale:")
    print(num_cols)
    print("Categorical columns to encode:")
    print(categorical_cols)

    # -----------------------------
    # Separate ID and target
    # -----------------------------
    ids = df[[id_col]].copy()
    y = df[[target_col]].copy()
    X = df.drop(columns=[id_col, target_col]).copy()

    # -----------------------------
    # Apply same fixed-rule preprocessing
    # -----------------------------
    
    X["age_group"] = pd.cut(
        X["age"],
        bins=[0, 18, 25, 40, 65, 80, np.inf],
        labels=["0-18", "19-25", "26-40", "41-65", "66-80", "81+"],
        right=False,
    )
    X = X.drop(columns=["age"])

    X["medications_prescribed"] = X["medications_prescribed"].replace("", pd.NA)
    X["medications_prescribed"] = X["medications_prescribed"].astype(float)

    X["medications_prescribed"] = X["medications_prescribed"].apply(
        lambda x: 1 if x > 0 else 0
    )

    X["number_of_prior_visits"] = X["number_of_prior_visits"].replace("", pd.NA)
    X["number_of_prior_visits"] = X["number_of_prior_visits"].astype(float)

    X["length_of_stay_score"] = X["length_of_stay"].apply(
        lambda x: (
            1
            if x <= 1
            else (
                2
                if x <= 2
                else (3 if x <= 3 else (4 if x <= 6 else (5 if x <= 14 else 7)))
            )
        )
    )
    X.drop(columns=["length_of_stay"], inplace=True)

    # -----------------------------
    # Apply learned preprocessing
    # -----------------------------

    # Impute and scale numerics
    X[num_cols] = median_imputer.transform(X[num_cols])
    X[num_cols] = scaler.transform(X[num_cols])
    
    # impute categories
    X[categorical_cols] = mode_imputer.transform(X[categorical_cols])

    #encode
    X = pd.get_dummies(X, drop_first=True) #this may need to be changed in the future to ensure consistency but idk
    

    # -----------------------------
    # Align validation columns to training columns
    # -----------------------------
    # Important safeguard: ensure exact feature-space match with training output
    X_prepared = X.reindex(columns=expected_output_cols, fill_value=0)

    # -----------------------------
    # Reconstruct final validation dataset
    # -----------------------------
    prepared_df = pd.concat([ids, X_prepared, y], axis=1)

    print(f"Prepared validation dataset shape: {prepared_df.shape}")

    # -----------------------------
    # Save output dataset
    # -----------------------------
    prepared_df.to_csv(output_dataset.path, index=False)

### Component 6 — `train_model`

Trains a `RandomForestClassifier` on the preprocessed training data and saves the model artifact + metadata.

In [ ]:
@component(
    packages_to_install=[
        "pandas",
        "numpy",
        "scikit-learn",
        "joblib"
    ]
)
def train_model(
    input_dataset: Input[Dataset],
    model_artifact: Output[Artifact],
    target_col: str = "target",
    id_col: str = "patient_id",
    n_estimators: int = 250,
    max_depth: int = 8,
    random_state: int = 42
):
    import os
    import json
    import joblib
    import pandas as pd
    from sklearn.ensemble import RandomForestClassifier

    # -----------------------------
    # Load prepared training dataset
    # -----------------------------
    df = pd.read_csv(input_dataset.path)

    print(f"Training dataset shape: {df.shape}")


    # -----------------------------
    # Separate features and target
    # -----------------------------
    feature_cols = [col for col in df.columns if col not in [id_col, target_col]]

    if len(feature_cols) == 0:
        raise ValueError("No feature columns available for model training")

    X_train = df[feature_cols]
    y_train = df[target_col]

    print(f"Number of training rows: {len(df)}")
    print(f"Number of features: {len(feature_cols)}")

    print("Training target distribution:")
    print(y_train.value_counts(dropna=False))

    # -----------------------------
    # Fit model
    # -----------------------------
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=random_state,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    print("Model training complete.")

    # -----------------------------
    # Save model artifact
    # -----------------------------
    os.makedirs(model_artifact.path, exist_ok=True)

    joblib.dump(model, os.path.join(model_artifact.path, "model.joblib"))

    metadata = {
        "model_type": "RandomForestClassifier",
        "target_col": target_col,
        "id_col": id_col,
        "feature_cols": feature_cols,
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "random_state": random_state,
        "n_training_rows": int(len(df)),
        "n_features": int(len(feature_cols))
    }

    with open(os.path.join(model_artifact.path, "model_metadata.json"), "w") as f:
        json.dump(metadata, f, indent=2)

    print("Saved model artifact contents:")
    print(os.listdir(model_artifact.path))

### Component 7 — `evaluate_model`

Runs inference on the validation split, logs F1/precision/recall/accuracy to Vertex Metrics, and saves a predictions CSV and evaluation report.

In [ ]:
@component(
    packages_to_install=[
        "pandas",
        "numpy",
        "scikit-learn",
        "joblib"
    ]
)
def evaluate_model(
    input_dataset: Input[Dataset],
    model_artifact: Input[Artifact],
    metrics: Output[Metrics],
    predictions_dataset: Output[Dataset],
    evaluation_report: Output[Artifact],
    target_col: str = "target",
    id_col: str = "patient_id"
):
    import os
    import json
    import joblib
    import pandas as pd
    from sklearn.metrics import (
        f1_score,
        precision_score,
        recall_score,
        confusion_matrix,
        accuracy_score
    )

    # -----------------------------
    # Load validation dataset
    # -----------------------------
    df = pd.read_csv(input_dataset.path)

    print(f"Validation dataset shape: {df.shape}")

    # -----------------------------
    # Load trained model + metadata
    # -----------------------------
    model_path = os.path.join(model_artifact.path, "model.joblib")
    metadata_path = os.path.join(model_artifact.path, "model_metadata.json")

    if not os.path.exists(model_path):
        raise ValueError(f"Missing model file: {model_path}")

    if not os.path.exists(metadata_path):
        raise ValueError(f"Missing model metadata file: {metadata_path}")

    model = joblib.load(model_path)

    with open(metadata_path, "r") as f:
        model_metadata = json.load(f)

    feature_cols = model_metadata["feature_cols"]

    # -----------------------------
    # Build validation feature matrix
    # -----------------------------
    X_val = df.reindex(columns=feature_cols, fill_value=0)
    y_val = df[target_col]

    print(f"Number of validation rows: {len(df)}")
    print(f"Number of model features expected: {len(feature_cols)}")

    # -----------------------------
    # Predict
    # -----------------------------
    y_pred = model.predict(X_val)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_val)[:, 1]
    else:
        y_prob = None

    # -----------------------------
    # Compute evaluation metrics
    # -----------------------------
    f1 = f1_score(y_val, y_pred, zero_division=0)
    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    accuracy = accuracy_score(y_val, y_pred)

    cm = confusion_matrix(y_val, y_pred)
    tn, fp, fn, tp = cm.ravel()

    print("Evaluation metrics:")
    print(f"F1 Score:   {f1:.4f}")
    print(f"Precision:  {precision:.4f}")
    print(f"Recall:     {recall:.4f}")
    print(f"Accuracy:   {accuracy:.4f}")
    print("Confusion Matrix:")
    print(cm)

    # -----------------------------
    # Log metrics for Vertex / KFP
    # -----------------------------
    metrics.log_metric("f1_score", float(f1))
    metrics.log_metric("precision", float(precision))
    metrics.log_metric("recall", float(recall))
    metrics.log_metric("accuracy", float(accuracy))
    metrics.log_metric("true_negatives", int(tn))
    metrics.log_metric("false_positives", int(fp))
    metrics.log_metric("false_negatives", int(fn))
    metrics.log_metric("true_positives", int(tp))

    # -----------------------------
    # Save predictions dataset
    # -----------------------------
    predictions_df = pd.DataFrame({
        id_col: df[id_col],
        "actual": y_val,
        "predicted": y_pred
    })

    if y_prob is not None:
        predictions_df["predicted_probability"] = y_prob

    predictions_df.to_csv(predictions_dataset.path, index=False)

    # -----------------------------
    # Save evaluation report artifact
    # -----------------------------
    os.makedirs(evaluation_report.path, exist_ok=True)

    report = {
        "model_type": model_metadata.get("model_type", "unknown"),
        "n_validation_rows": int(len(df)),
        "n_features_used": int(len(feature_cols)),
        "metrics": {
            "f1_score": float(f1),
            "precision": float(precision),
            "recall": float(recall),
            "accuracy": float(accuracy)
        },
        "confusion_matrix": {
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp)
        }
    }

    with open(os.path.join(evaluation_report.path, "evaluation_report.json"), "w") as f:
        json.dump(report, f, indent=2)

    print("Saved evaluation report contents:")
    print(os.listdir(evaluation_report.path))

---
## 7. Pipeline Assembly & Deployment

Assembles the components into a KFP pipeline, compiles to JSON, and submits to Vertex AI Pipelines.

In [ ]:
from kfp.v2 import dsl
from kfp.dsl import pipeline


@pipeline(name="1.4.x-readmissions-smote-training-pipeline")
def readmissions_smote_training_pipeline(training_dataset_path: str):

    # 1. load data set
    validated_data_task = load_validate_data(input_dataset_path=training_dataset_path)

    # 2. split dataset
    split_task = split_data(input_dataset=validated_data_task.outputs["output_dataset"])

    # 3. oversample training
    oversampled_train_task = oversample_training(
        input_dataset=split_task.outputs["train_dataset"]
    )

    # 4. preprocess training
    preprocessed_train_task = fit_apply_preprocessing(
        input_dataset=oversampled_train_task.outputs["output_dataset"],
    )

    # 5. preprocess validation
    preprocessed_validation_task = apply_preprocessing(
        input_dataset=split_task.outputs["validation_dataset"],
        preprocessing_artifacts=preprocessed_train_task.outputs[
            "preprocessing_artifacts"
        ],
    )

    # 6. train and fit model
    trained_model_task = train_model(
        input_dataset=preprocessed_train_task.outputs["output_dataset"]
    )

    # 7. eval model
    evaluate_model(
        input_dataset=preprocessed_validation_task.outputs["output_dataset"],
        model_artifact=trained_model_task.outputs["model_artifact"],
    )

In [ ]:
from kfp import compiler
TRAINING_PIPELINE_JSON = "readmissions_smote_training_pipeline.json"

compiler.Compiler().compile(
    pipeline_func=readmissions_smote_training_pipeline,
    package_path=TRAINING_PIPELINE_JSON
)

In [ ]:
dataset_folder_uri = f"{BUCKET_ROOT_URI}/datasets/readmissions"
artifact_base_uri = f"{BUCKET_ROOT_URI}/training_pipeline_artifacts"
model_version = "v0"

In [ ]:
from time import time


training_dataset_path = f"{dataset_folder_uri}/v1.4/train.csv"

training_job = aiplatform.PipelineJob(
    job_id=f"readmissions-smote-training-pipeline-{model_version}-{int(time())}",
    display_name=f"readmissions-1.4-smote-{model_version}",
    template_path=TRAINING_PIPELINE_JSON,
    pipeline_root=BUCKET_ROOT_URI,
    parameter_values={
        "training_dataset_path": training_dataset_path,
    },
    enable_caching=True,
    project=PROJECT_ID,
    location=LOCATION
)

training_job.submit(
)



In [ ]:
log_pipeline_run(
    pipeline_job=training_job,
    dataset_version="v1.4.0",
    training_dataset_path=training_dataset_path,
    model_version=model_version,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    wait_for_completion=True,
)

---
## 8. 1.4.1 Dataset Model Comparisons 

Experimenting on what model to use on the 1.4.1 Dataset
- 1.4.1 is the name of the dataset created from the pipeline above, which includes feature engineering and preprocessing steps.

In [ ]:
# run pipeline locally to get preprocessed data
import kfp.local

# Initialize local runner (SubprocessRunner installs packages_to_install into a venv)
kfp.local.init(runner=kfp.local.SubprocessRunner(use_venv=True))

# Call the pipeline function directly — no PipelineJob, no GCS needed
readmissions_smote_training_pipeline(
    training_dataset_path="../data/interim/1.4-edited-data.csv"
)

In [ ]:
train_df = pd.read_csv(
    "./local_outputs/1-4-x-readmissions-smote-training-pipeline-2026-05-01-14-32-53-985651/fit-apply-preprocessing/output_dataset"
)
val_df = pd.read_csv(
    "./local_outputs/1-4-x-readmissions-smote-training-pipeline-2026-05-01-14-32-53-985651/apply-preprocessing/output_dataset"
)


---
## 9. Model Registry & Managed Datasets

Extends the training pipeline with two additional steps:

1. **`register_model`** (KFP component) — uploads the trained model artifact to Vertex AI Model Registry for versioning, lineage tracking, and eventual endpoint deployment.
2. **Dataset registration** (notebook cell) — registers the training CSV in Vertex AI Managed Datasets so it appears in the console alongside the model, enabling lineage from dataset → pipeline run → model version.

The updated pipeline is assembled as `readmissions_registry_pipeline` below.

### Component 8 — `register_model`

Uploads the trained model artifact (GCS directory) to Vertex AI Model Registry.
- Accepts an optional `serving_container_image_uri`; defaults to the pre-built sklearn 1.5 CPU image.

In [ ]:
@component(
    packages_to_install=[
        "google-cloud-aiplatform",
        "joblib",
    ]
)
def register_model(
    model_artifact: Input[Artifact],
    project: str,
    location: str,
    model_display_name: str,
    training_dataset_path: str,
    serving_container_image_uri: str = "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-5:latest",
):
    """Uploads the trained model artifact directory to Vertex AI Model Registry.

    Attaches dataset source and model hyperparameters as labels.
    """
    import json
    import os
    import re
    from google.cloud import aiplatform

    def _to_label_val(value: str) -> str:
        """Sanitize a string to a valid GCP label value (lowercase, [a-z0-9_-], max 63 chars)."""
        value = str(value).lower()
        value = re.sub(r"[^a-z0-9_\-]", "-", value)
        value = re.sub(r"-+", "-", value).strip("-")
        return value[:63]

    aiplatform.init(project=project, location=location)

    print(f"Artifact URI: {model_artifact.uri}")
    print(f"Registering model: {model_display_name}")

    # ----------------------------------------
    # Build labels from model metadata + dataset
    # ----------------------------------------
    labels = {}

    # Dataset source: extract meaningful path segment (e.g. "datasets/readmissions/v1-4/train")
    dataset_label = _to_label_val(
        "/".join(training_dataset_path.replace("gs://", "").split("/")[1:])
    )
    labels["dataset-source"] = dataset_label[:63]

    # Model hyperparameters from saved metadata
    metadata_path = os.path.join(model_artifact.path, "model_metadata.json")

    if os.path.exists(metadata_path):
        with open(metadata_path) as f:
            meta = json.load(f)

        labels["model-type"] = _to_label_val(meta.get("model_type", "unknown"))
        labels["n-features"] = _to_label_val(meta.get("n_features", ""))
        labels["n-training-rows"] = _to_label_val(meta.get("n_training_rows", ""))
        labels["max-depth"] = _to_label_val(meta.get("max_depth", ""))
        labels["n-estimators"] = _to_label_val(meta.get("n_estimators", ""))

    print(f"Labels: {labels}")

    registered_model = aiplatform.Model.upload(
        display_name=model_display_name,
        artifact_uri=model_artifact.uri,
        serving_container_image_uri=serving_container_image_uri,
        labels=labels,
        sync=True,
    )

    print(f"Model registered: {registered_model.resource_name}")


### Pipeline — `readmissions_registry_pipeline`

Same steps as the original pipeline, with `register_model` added as step 8 after evaluation.

In [ ]:
from kfp.v2 import dsl
from kfp.dsl import pipeline


@pipeline(name="1.4.x-readmissions-registry-pipeline")
def readmissions_registry_pipeline(
    training_dataset_path: str,
    project: str,
    location: str,
    model_display_name: str,
):
    # 1. load & validate
    validated_data_task = load_validate_data(input_dataset_path=training_dataset_path)

    # 2. split
    split_task = split_data(input_dataset=validated_data_task.outputs["output_dataset"])

    # 3. oversample training
    oversampled_train_task = oversample_training(
        input_dataset=split_task.outputs["train_dataset"]
    )

    # 4. fit preprocessing on training split
    preprocessed_train_task = fit_apply_preprocessing(
        input_dataset=oversampled_train_task.outputs["output_dataset"],
    )

    # 5. apply preprocessing to validation split
    preprocessed_validation_task = apply_preprocessing(
        input_dataset=split_task.outputs["validation_dataset"],
        preprocessing_artifacts=preprocessed_train_task.outputs["preprocessing_artifacts"],
    )

    # 6. train
    trained_model_task = train_model(
        input_dataset=preprocessed_train_task.outputs["output_dataset"]
    )

    # 7. evaluate
    eval_task = evaluate_model(
        input_dataset=preprocessed_validation_task.outputs["output_dataset"],
        model_artifact=trained_model_task.outputs["model_artifact"],
    )

    # 8. register model in Vertex AI Model Registry
    register_model(
        model_artifact=trained_model_task.outputs["model_artifact"],
        project=project,
        location=location,
        model_display_name=model_display_name,
        training_dataset_path=training_dataset_path,
    )

In [ ]:
REGISTRY_PIPELINE_JSON = "readmissions_registry_pipeline.json"

compiler.Compiler().compile(
    pipeline_func=readmissions_registry_pipeline,
    package_path=REGISTRY_PIPELINE_JSON,
)

In [ ]:
registry_model_version = "v1"
registry_model_display_name = f"readmissions-rf-{registry_model_version}"
registry_training_dataset_path = f"{dataset_folder_uri}/v1.4/train.csv"

registry_job = aiplatform.PipelineJob(
    job_id=f"readmissions-registry-pipeline-{registry_model_version}-{int(time())}",
    display_name=f"readmissions-registry-{registry_model_version}",
    template_path=REGISTRY_PIPELINE_JSON,
    pipeline_root=BUCKET_ROOT_URI,
    parameter_values={
        "training_dataset_path": registry_training_dataset_path,
        "project": PROJECT_ID,
        "location": LOCATION,
        "model_display_name": registry_model_display_name,
    },
    enable_caching=True,
    project=PROJECT_ID,
    location=LOCATION,
)

registry_job.submit()

### Register Training Dataset in Vertex AI Managed Datasets

Registers the GCS training CSV as a `TabularDataset` in Vertex AI. This creates a named, versioned dataset entry in the console (**Vertex AI → Datasets**) that can be linked to training jobs and model versions for lineage tracking.

Run this once per dataset version — it is idempotent in the sense that calling it again will just create a new entry with the same source URI.

In [ ]:
# Register the v1.4 training CSV as a Vertex AI Managed Tabular Dataset.
# The dataset points to the GCS URI — Vertex does not copy the data.
managed_dataset = aiplatform.TabularDataset.create(
    display_name="readmissions-training-v1.4",
    gcs_source=[f"{dataset_folder_uri}/v1.4/train.csv"],
    project=PROJECT_ID,
    location=LOCATION,
    labels={
        "version": "v1-4",
        "pipeline": "readmissions-registry",
        "split": "train",
    },
    sync=True,
)

print(f"Managed dataset resource name: {managed_dataset.resource_name}")
print(f"Console URL: https://console.cloud.google.com/vertex-ai/datasets/{managed_dataset.name}?project={PROJECT_ID}")